In [7]:
import polars as pl
import os
import lightgbm as lgb

In [21]:
articles_path='./data/articles.parquet'
transaction_path='./data/transactions.parquet'
articles=pl.read_parquet(articles_path)
transactions=pl.read_parquet(transaction_path)
data=pl.read_parquet('recall.parquet')

In [9]:
DAY = 86400
WEEK = 7 * DAY

max_time = transactions['time'].max()         # 模拟“下一周”
min_time = max_time - 10 * WEEK      # 只保留最近 10 周

valid_start = max_time - WEEK      # 最近一周做验证
valid_end   = max_time

transactions = transactions.filter(pl.col("time") > min_time)

valid = transactions.filter(
    (pl.col("time") > valid_start) & (pl.col("time") <= valid_end)
)

In [10]:
gt = (
    valid
    .select(["customer_id", "article_id"])
    .with_columns(pl.lit(1).alias("label"))
)


In [22]:
data = (
    data
    .join(
        gt,
        on=["customer_id", "article_id"],
        how="left"
    )
    .with_columns(
        pl.col("label").fill_null(0)
    )
)


In [23]:
data = data.sort(["customer_id", "article_id"])

group = (
    data
    .group_by("customer_id", maintain_order=True)
    .len()
    .select("len")
    .to_numpy()
    .flatten()
)


In [13]:
articles.columns


['article_id',
 'product_code',
 'product_type_no',
 'graphical_appearance_no',
 'colour_group_code',
 'perceived_colour_value_id',
 'perceived_colour_master_id',
 'department_no',
 'index_group_no',
 'section_no',
 'garment_group_no',
 'index_code_A',
 'index_code_B',
 'index_code_C',
 'index_code_D',
 'index_code_F',
 'index_code_G',
 'index_code_H',
 'index_code_I',
 'index_code_J',
 'index_code_S']

In [14]:
features = [
     'itemcf_score',
     'w2v_score',
     'from_itemcf',
     'from_w2vec',
     'from_popularity_w1',
     'from_popularity_w2',
     'from_popularity_w3',
     'from_popularity_w4',
     'from_repurchase',
     'product_code',
     'product_type_no',
     'graphical_appearance_no',
     'colour_group_code',
     'perceived_colour_value_id',
     'perceived_colour_master_id',
     'department_no',
     'index_code_A',
     'index_code_B',
     'index_code_C',
     'index_code_D',
     'index_code_F',
     'index_code_G',
     'index_code_H',
     'index_code_I',
     'index_code_J',
     'index_code_S',
     'index_group_no',
     'section_no',
     'garment_group_no',
     'FN',
     'Active',
     'fashion_news_frequency_Monthly',
     'fashion_news_frequency_NONE',
     'fashion_news_frequency_Regularly',
     'club_member_status_ACTIVE',
     'club_member_status_LEFT CLUB',
     'club_member_status_PRE-CREATE',
     'age_0',
     'age_1',
     'age_2',
     'recall_cnt'
]

X = data.select(features).to_pandas()
y = data["label"].to_pandas()


In [15]:
del data,transactions,articles

In [16]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="map",
    eval_at=[12],
    n_estimators=200,
    learning_rate=0.05,
)

In [17]:
ranker.fit(
    X,
    y,
    group=group,
)

/usr/local/miniconda3/envs/py312/lib/python3.12/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Total groups: 410111, total data: 62871693
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 3.897043 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1334
[LightGBM] [Info] Number of data points in the train set: 62871693, number of used features: 41


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,200
,subsample_for_bin,200000
,objective,'lambdarank'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [18]:
# 训练完成后
booster = ranker.booster_
booster.save_model("lgb_ranker.txt")

In [19]:
booster = lgb.Booster(model_file="lgb_ranker.txt")

In [24]:
data = data.with_columns(
    pl.Series("pred", booster.predict(X))
)

In [25]:
pred_df = (
    data
    .sort(["customer_id", "pred"], descending=[False, True])
    .group_by("customer_id")
    .agg(pl.col("article_id").head(12).alias("pred_items"))
)


In [26]:
gt_df = (
    valid
    .group_by("customer_id")
    .agg(pl.col("article_id").unique().alias("gt_items"))
)


In [27]:
from numpy import mean

def apk(actual, predicted, k=12):
    actual = actual.to_list()
    predicted = predicted.to_list()

    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0
    hits = 0
    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i + 1)

    return score / min(len(actual), k) if len(actual) > 0 else 0.0


eval_df = pred_df.join(gt_df, on="customer_id", how="inner")

map12 = mean([
    apk(gt, pred, 12)
    for gt, pred in zip(eval_df["gt_items"], eval_df["pred_items"])
])

print("MAP@12 =", map12)


MAP@12 = 0.04053328771864862
